# 04 LiteLLM price and capability JSON: model_price.csv

**Purpose.** This notebook downloads the LiteLLM community price file and flattens it into `model_price.csv`. Q1 uses it for the price and cost-of-switching side, and for the list of cloud hosts (Bedrock, Azure, Vertex, Fireworks and others) that offer the same model. It also shows which hosted models support function calling, vision and reasoning.

**Question served.** Q1 (price and hosting), with support for Q3 (cost) if that chapter is kept.

**How to run.** Run all cells from top to bottom. The file is about 2.9 MB.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one JSON file |
| Where it comes from | BerriAI LiteLLM repository. Repository https://github.com/BerriAI/litellm . File page https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json . Download address https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json |
| Licence and authorisation | MIT, except the `enterprise/` directory, which we do not use. LICENSE https://github.com/BerriAI/litellm/blob/main/LICENSE . No account or key. Details in Section 1 |
| Download size | about 2.9 MB (4,207 keys on 2026-10-18) |
| Expected time | download 2 to 10 seconds, whole notebook under 20 seconds. Measured in our test run: 2.0 seconds including a real download |
| Data as of | the day you run it (the file carries no date of its own). See `processed/data_currency.csv` |
| To update | Run the notebook again |

## 1. Source, reference links and authorisation

| Item | Detail |
|---|---|
| Publisher | BerriAI, maintainers of LiteLLM, with community contributions |
| Dataset | `model_prices_and_context_window.json`, a JSON object with one key per model entry |
| Repository | https://github.com/BerriAI/litellm |
| File page | https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json |
| Download URL used | https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json (this is the raw form of the file page, the same URL also works as https://github.com/BerriAI/litellm/raw/refs/heads/main/model_prices_and_context_window.json from a browser) |
| Licence | MIT. The repository LICENSE says "Portions of this software are licensed as follows". The MIT terms cover everything **except** the `enterprise/` directory, which has its own licence. The price JSON sits at the repository root, so it falls under MIT. LICENSE page: https://github.com/BerriAI/litellm/blob/main/LICENSE . Checked 18 Oct 2026 |
| What MIT asks | Keep the copyright and licence notice. The software has no warranty. See https://choosealicense.com/licenses/mit/ |
| Authorisation | No account, key or login needed. Public file. The notebook makes one request. |
| Data quality note | Prices are entered by the community and by LiteLLM maintainers from provider price pages. The file is not an official price list from any provider. The `source` field on many entries points to the provider page. Check a price against that page before quoting it as fact. |

## 2. Table structure

### 2.1 Structure of the source file

A single JSON object. Each key is a model entry such as `azure_ai/FW-Kimi-K3` or `bedrock/ap-northeast-1/moonshotai.kimi-k2.5`. Each value is an object with these common fields (counts from the snapshot of 18 Oct 2026, 4,207 keys):

| Field | Meaning | Present in about |
|---|---|---|
| `litellm_provider` | the host or API family, for example `openai`, `bedrock`, `azure_ai`, `dashscope`, `gemini` | all entries |
| `mode` | chat, embedding, image_generation and others | almost all |
| `input_cost_per_token`, `output_cost_per_token` | US dollars per token | 3,393 and 3,385 entries |
| `cache_read_input_token_cost` | US dollars per cached input token | 1,494 |
| `max_input_tokens`, `max_output_tokens` | context limits | 3,363 and 2,904 |
| `supports_function_calling`, `supports_vision`, `supports_reasoning`, `supports_prompt_caching` | booleans | 1,200 to 2,500 |
| `supported_regions` | list of regions, only on some entries | few |
| `deprecation_date` | text date | 804 |
| `source` | URL of the provider price page | 2,887 |

The first key, `sample_spec`, is a documentation template and is not a model. The notebook drops it.

Since November 2026 the file also holds two kinds of entry that are not models. About a hundred are pointers of the form `{"alias_of": "<another key>"}` (for example `zai/glm-latest` points to `zai/glm-5.3`), and one block, `fallback_generalizations`, holds routing rules. Neither has a host, a mode or a price. The notebook drops them like `sample_spec`, counts them, and stops if an entry without a host is neither of the two kinds.

### 2.2 Structure of the output `processed/model_price.csv`

One row per model entry (minus `sample_spec`, the pointer entries and the routing block).

| Output column | Type | How it is made |
|---|---|---|
| `model_key` | text | the JSON key, unique |
| `litellm_provider` | text | as given |
| `key_prefix` | text | the part of the key before the first slash, empty if there is none. Usually a host or region, but sometimes an image-size or step tag, so use `litellm_provider` for the host |
| `model_stem` | text | the part after the last slash, lower case. The best join key to other tables, still needs a crosswalk |
| `mode` | text | as given |
| `input_usd_per_million`, `output_usd_per_million`, `cache_read_usd_per_million` | float | per-token price times 1,000,000 |
| `max_input_tokens`, `max_output_tokens` | float | as given |
| `supports_function_calling`, `supports_vision`, `supports_reasoning`, `supports_prompt_caching` | bool or empty | as given |
| `supported_regions` | text | list joined with a vertical bar, empty if not given |
| `deprecation_date` | text | as given |
| `source_url` | text | the `source` field |
| `family` | text | kimi, deepseek, glm, qwen, minimax, or empty. Found by a text match on the key, see Section 6 |

## 3. Set up the paths and download helper

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "litellm_prices"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER}. Current folder: {cwd}")

MANUAL = HERE / "manual"                       # hand-downloaded source files (optional)
STORY_DATA = HERE.parents[1] / "data"          # the story's own data folder. The notebook never writes here.
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)
print("Story data     :", STORY_DATA, "(copy files here by hand, see the last section)")


HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
REFRESH = True  # False: reuse the file already in raw/ (same snapshot as the A3P1 app). True: try a fresh download first.


def record(source_id: str, url: str, path: Path, how: str) -> None:
    """Add (or replace) the retrieval-log entry for one raw file."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print(f"logged {source_id}: {len(content):,} bytes via {how}")


def get_file(source_id: str, url: str, target: str, manual_name: str = None) -> Path:
    """Return the path of a raw file. Order: fresh download, earlier download in raw/, manual copy."""
    path = RAW / target
    if REFRESH or not path.exists():
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as err:
            print(f"Download of {source_id} failed: {err}")
    manual = MANUAL / (manual_name or target)
    candidates = [p for p in (path, manual) if p.exists()]
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
        if best == path:
            print(f"Using the earlier download {path.name}")
        else:
            shutil.copy2(best, path)
            record(source_id, url, path, f"copied from manual folder {manual.parent.name}")
        return path
    raise FileNotFoundError(
        f"Could not get {source_id}. Open {url} in a browser, save the file as '{manual_name or target}' "
        f"inside {MANUAL}, then run this cell again.")

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\1a_ranked_shortlist\notebooks\litellm_prices
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\1a_ranked_shortlist\notebooks\litellm_prices\manual
Story data     : d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\1a_ranked_shortlist\data (copy files here by hand, see the last section)


## 4. Acquire the file

In [2]:
PRICE_URL = "https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json"
price_path = get_file("litellm_prices", PRICE_URL, "model_prices_and_context_window.json")
prices = json.loads(price_path.read_text(encoding="utf-8"))
print(type(prices).__name__, "with", len(prices), "keys")

logged litellm_prices: 4,054,495 bytes via downloaded
dict with 5943 keys


## 5. Inspect

In [3]:
assert isinstance(prices, dict), "LiteLLM changed the file layout. Update Section 2."
print("Is sample_spec present:", "sample_spec" in prices)
entries_all = {k: v for k, v in prices.items() if k != "sample_spec" and isinstance(v, dict)}
# Newer versions of the file also hold entries that are not models: pointers such as
# {"alias_of": "<another key>"} and a routing block ("fallback_generalizations" with "rules").
# They have no host, no mode and no price, so they are kept out of the table.
NON_MODEL_FIELDS = {"alias_of", "rules"}
non_models = {k: v for k, v in entries_all.items() if "litellm_provider" not in v}
unknown_non_models = [k for k, v in non_models.items() if not set(v) <= NON_MODEL_FIELDS]
aliases = [k for k, v in non_models.items() if "alias_of" in v]
entries = {k: v for k, v in entries_all.items() if k not in non_models}
print("Entries in the file (without sample_spec):", len(entries_all))
print("Pointer entries (alias_of):", len(aliases), "| other non-model entries:", len(non_models) - len(aliases))
print("Model entries:", len(entries))

from collections import Counter
print()
print("Most common hosts:")
print(Counter(v.get("litellm_provider") for v in entries.values()).most_common(12))
print()
print("Modes:")
print(Counter(v.get("mode") for v in entries.values()).most_common(8))

Is sample_spec present: True
Entries in the file (without sample_spec): 5942
Pointer entries (alias_of): 107 | other non-model entries: 1
Model entries: 5834

Most common hosts:
[('fal_ai', 914), ('fireworks_ai', 356), ('bedrock', 342), ('replicate', 323), ('azure', 322), ('bedrock_converse', 250), ('openai', 223), ('azure_ai', 140), ('deepinfra', 136), ('novita', 135), ('gmi', 121), ('vercel_ai_gateway', 99)]

Modes:
[('chat', 3560), ('image_generation', 993), ('video_generation', 445), ('responses', 206), ('embedding', 157), ('audio_transcription', 113), ('audio_speech', 81), ('realtime', 59)]


## 6. Flatten into a table

The price in the file is dollars per token. We multiply by one million so that numbers read as dollars per million tokens, the unit used on provider price pages.

In [4]:
FAMILY_PATTERNS = {
    "kimi": "kimi|moonshot",
    "deepseek": "deepseek",
    "glm": "glm|zhipu|z-ai|zai-org",
    "qwen": "qwen|dashscope",
    "minimax": "minimax",
}

def per_million(value):
    return value * 1_000_000 if isinstance(value, (int, float)) else None

def family_of(key: str):
    low = key.lower()
    for name, pattern in FAMILY_PATTERNS.items():
        if any(part in low for part in pattern.split("|")):
            return name
    return None

rows = []
for key, v in entries.items():
    regions = v.get("supported_regions")
    rows.append({
        "model_key": key,
        "litellm_provider": v.get("litellm_provider"),
        "key_prefix": key.split("/")[0] if "/" in key else None,
        "model_stem": key.split("/")[-1].lower(),
        "mode": v.get("mode"),
        "input_usd_per_million": per_million(v.get("input_cost_per_token")),
        "output_usd_per_million": per_million(v.get("output_cost_per_token")),
        "cache_read_usd_per_million": per_million(v.get("cache_read_input_token_cost")),
        "max_input_tokens": v.get("max_input_tokens"),
        "max_output_tokens": v.get("max_output_tokens"),
        "supports_function_calling": v.get("supports_function_calling"),
        "supports_vision": v.get("supports_vision"),
        "supports_reasoning": v.get("supports_reasoning"),
        "supports_prompt_caching": v.get("supports_prompt_caching"),
        "supported_regions": "|".join(regions) if isinstance(regions, list) else None,
        "deprecation_date": v.get("deprecation_date"),
        "source_url": v.get("source"),
        "family": family_of(key),
    })
price_table = pd.DataFrame(rows)
print(price_table.shape)
price_table.head()

(5834, 18)


,model_key,litellm_provider,key_prefix,model_stem,mode,input_usd_per_million,output_usd_per_million,cache_read_usd_per_million,max_input_tokens,max_output_tokens,supports_function_calling,supports_vision,supports_reasoning,supports_prompt_caching,supported_regions,deprecation_date,source_url,family
0,azure/gpt-realtime-2.1-2026-07-07,azure,azure,gpt-realtime-2.1-2026-07-07,realtime,4.0,24.0,0.40,32000.0,4096.0,True,None,True,True,None,2027-06-25,https://learn.microsoft.com/en-us/azure/foundr...,None
1,azure/gpt-realtime-2.1-mini-2026-07-07,azure,azure,gpt-realtime-2.1-mini-2026-07-07,realtime,0.6,2.4,0.06,32000.0,4096.0,True,None,True,True,None,2027-06-25,https://learn.microsoft.com/en-us/azure/foundr...,None
2,azure/gpt-realtime-translate-2026-05-06,azure,azure,gpt-realtime-translate-2026-05-06,realtime,NaN,NaN,NaN,32000.0,4096.0,None,None,None,None,None,None,https://techcommunity.microsoft.com/blog/azure...,None
3,azure/gpt-realtime-translate-2026-05-07,azure,azure,gpt-realtime-translate-2026-05-07,realtime,NaN,NaN,NaN,32000.0,4096.0,None,None,None,None,None,None,https://ai.azure.com/catalog/models/gpt-realti...,None
4,azure/gpt-realtime-whisper-2026-05-06,azure,azure,gpt-realtime-whisper-2026-05-06,audio_transcription,NaN,NaN,NaN,32000.0,4096.0,None,None,None,None,None,None,https://learn.microsoft.com/en-us/azure/foundr...,None


Keyword matching on the key is a rough filter. For example "glm" also catches names that merely contain those letters. The `family` column is a convenience for browsing and must not be used as a final classification. The model crosswalk step does the proper match.

## 7. Validate

In [5]:
problems = []
if price_table["model_key"].duplicated().any():
    problems.append("duplicated model_key")
if price_table["litellm_provider"].isna().mean() > 0.01:
    problems.append("many entries without litellm_provider")
if unknown_non_models:
    problems.append("entries without litellm_provider that are neither alias_of pointers nor rules: " + ", ".join(unknown_non_models[:5]))
priced = price_table.dropna(subset=["input_usd_per_million"])
if (priced["input_usd_per_million"] < 0).any():
    problems.append("negative price")
if len(priced) < 1000:
    problems.append("fewer than 1000 priced entries, the file layout may have changed")
# Prices above $1,000 per million tokens are almost certainly unit mistakes.
extreme = priced[priced["input_usd_per_million"] > 1000]
print("Entries priced above $1,000 per million input tokens (probably data-entry errors, check before using):", len(extreme))
print(extreme[["model_key", "mode", "input_usd_per_million"]].head(5).to_string(index=False))
print()
print("Problems:", problems if problems else "none")
assert not problems, problems
print()
print("Priced entries:", len(priced), "of", len(price_table))
print("Chat entries with both prices:", int((price_table["mode"].eq("chat") & price_table["input_usd_per_million"].notna() & price_table["output_usd_per_million"].notna()).sum()))

Entries priced above $1,000 per million input tokens (probably data-entry errors, check before using): 0
Empty DataFrame
Columns: [model_key, mode, input_usd_per_million]
Index: []

Problems: none

Priced entries: 4030 of 5834
Chat entries with both prices: 3420


## 8. Save

In [6]:
price_table.to_csv(OUT / "model_price.csv", index=False, encoding="utf-8")
print("saved", OUT / "model_price.csv", price_table.shape)

saved d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\1a_ranked_shortlist\notebooks\litellm_prices\processed\model_price.csv (5834, 18)


## 9. Quick look: the five hedging-relevant families across hosts

How many price entries exist for each model family, and on how many different hosts. This shows how widely the models are offered through outside clouds. It is a count of file entries, not proof that a host serves a given region.

In [7]:
fam = price_table.dropna(subset=["family"])
summary = (fam[fam["mode"].eq("chat")]
           .groupby("family")
           .agg(entries=("model_key", "size"),
                hosts=("litellm_provider", "nunique"),
                median_input_usd_per_m=("input_usd_per_million", "median"),
                median_output_usd_per_m=("output_usd_per_million", "median")))
summary

,entries,hosts,median_input_usd_per_m,median_output_usd_per_m
family,,,,
deepseek,313,53,0.310010,1.2000
glm,208,43,1.000000,3.1750
kimi,169,39,0.950000,4.0000
minimax,95,25,0.300000,1.2000
qwen,465,46,0.225005,0.9782


## How to update, and what to do if the download is blocked

**Automatic.** Run all cells again. Section 4 downloads the raw JSON and replaces `raw/model_prices_and_context_window.json`. About 5 seconds.

**Manual download (use this if you see `Download of litellm_prices failed`).**

1. Open https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json in a browser. It shows a long page of text.
2. Press Ctrl+S and save the page as `model_prices_and_context_window.json` into `manual\`. Do not copy and paste the text, because a paste can break the file.
3. Another route: open the file page https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json and use the Raw button, then save it the same way.
4. Run the notebook again. If the download still fails, the notebook compares `raw/` with `manual/` and uses the **newer** file.

**Update check.** After any update, open `raw/retrieval_log.json` in this folder. The `how` field says `downloaded` when the fresh file was used and `copied from manual folder` when the hand-saved copy was used. Then open `processed/data_currency.csv` and confirm `extracted_utc` and `data_covers_to` are what you expect.

## 10. Run time and limits

**Time to extract.** One request for a file of about 2.9 MB. Expect 2 to 10 seconds for the download and 1 to 2 seconds for the flattening, so a full run takes **under 20 seconds**. The last cell prints the measured time.

**Limits and known risks.**

1. The file is a moving target. Every download is a different snapshot. The retrieval log in `raw/` records the time and hash.
2. Prices are community-entered. They can lag a provider's real price. Use them for comparison and cite the `source_url` for any single price you quote.
3. A model key names a *host and model*, not a model alone. `bedrock/ap-northeast-1/...` and `azure_ai/FW-...` are different offers of related models.
4. The file says nothing about whether a host accepts customers in Hong Kong or mainland China. That is Search C (notebook 03).
5. Names differ from Epoch and LMArena. The crosswalk is a separate step.
6. The `enterprise/` code in the same repository has a different licence. We use only the JSON file.

## 11. References

1. BerriAI, LiteLLM repository. https://github.com/BerriAI/litellm
2. BerriAI, price file. https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json
3. BerriAI, LICENSE. https://github.com/BerriAI/litellm/blob/main/LICENSE
4. Choose a License, MIT. https://choosealicense.com/licenses/mit/
5. Project note on licences: `05 Gemini Deep Research/Licence Check for Open-Weight Models (English and 繁體中文).md`

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [8]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


dep = pd.to_datetime(price_table["deprecation_date"], errors="coerce")
log = json.loads(LOG_PATH.read_text(encoding="utf-8"))["litellm_prices"]
write_currency("LiteLLM price file (model_price.csv)",
               None, log["retrieved_utc"][:10],
               f"file in raw/ ({log['how']}, retrieved {log['retrieved_utc']}, see raw/retrieval_log.json) from the live main branch. With REFRESH = False the notebook reuses this file, so the prices are as old as that retrieval date and not as old as this run. The file carries no date of its own. Deprecation dates inside it run from "
               f"{dep.min().date() if dep.notna().any() else 'n/a'} to {dep.max().date() if dep.notna().any() else 'n/a'}",
               "Community repository. The first commit date of the price file was not recorded in our notes.",
               "Continuous, many commits per week")

dataset                  LiteLLM price file (model_price.csv)
extracted_utc            2026-11-12T19:12:37Z
data_covers_from         not stated in the data
data_covers_to           2026-11-12
snapshot_note            file in raw/ (downloaded, retrieved 2026-11-12T19:12:35Z, see raw/retrieval_log.json) from the live main branch. With REFRESH = False the notebook reuses this file, so the prices are as old as that retrieval date and not as old as this run. The file carries no date of its own. Deprecation dates inside it run from 2025-06-16 to 2028-05-20
source_origin_note       Community repository. The first commit date of the price file was not recorded in our notes.
source_update_frequency  Continuous, many commits per week


In [9]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 2.7 seconds
Files written:
  processed\data_currency.csv  (676 bytes)
  processed\model_price.csv  (1,001,945 bytes)


## Copy the table into the story data folder (by hand)

The notebook does not write into the story's `data/` folder. Its outputs stay in `raw/` (files as downloaded) and `processed/` (tables it built). After a run, copy the file listed below from `processed/` into `../../data/` by hand. The page reads only that copy.

The table is kept as the notebook wrote it. It is not combined with the other sources here. The page does that. The cell below only prints. It shows which file to copy, its number of rows and whether the copy already in `data/` has the same content (line endings are ignored).

In [10]:
TO_COPY = ['model_price.csv']   # data_currency.csv stays here as a record of the run
for name in TO_COPY:
    made = OUT / name
    rows = len(pd.read_csv(made, keep_default_na=False))
    kept = STORY_DATA / name
    if not kept.exists():
        state = "not in the story data folder yet. Copy it."
    elif pd.read_csv(kept, keep_default_na=False).equals(pd.read_csv(made, keep_default_na=False)):
        state = "the copy in the story data folder has the same content. Nothing to do."
    else:
        state = "the copy in the story data folder is DIFFERENT. Copy the new file over it."
    print(f"{name}: {rows} rows. From {made.relative_to(HERE)} to {kept}. Status: {state}")


model_price.csv: 5834 rows. From processed\model_price.csv to d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\1a_ranked_shortlist\data\model_price.csv. Status: the copy in the story data folder is DIFFERENT. Copy the new file over it.
